# M2 — Government QA LoRA Model

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
!pip install -q "transformers>=4.43" "peft>=0.11.1" "accelerate>=0.30" "datasets>=2.19" "pandas>=2.2"

In [ ]:
import os, json, re, random
from pathlib import Path
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, get_peft_model, PeftModel
from safetensors.torch import load_file

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
try: torch.set_float32_matmul_precision("high")
except: pass

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
LORA_DIR = "/Volumes/main/default/thesis_project/M2_NoContext/G_M2_no_context.1.5"

tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"  # 推理左填充

# 关键：让 HF/accelerate 自动把权重放到 GPU（若必要会做分层映射）
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",            # ✅ 不要再 .to("cuda")
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base, LORA_DIR).eval()
model.config.use_cache = True

In [ ]:
tok.padding_side = "left"
MAX_IN_LEN = min(2048, getattr(model.config, "max_position_embeddings", 2048))
tok.model_max_length = MAX_IN_LEN

In [ ]:
#  读数据
import pandas as pd, numpy as np, os
from pathlib import Path
from tqdm.auto import tqdm
from math import ceil

EVAL_PATH= "/Volumes/main/default/thesis_project/GovernmentDocument/gov-report-qs/processed_20250804_234209/qs_test_qa_evidence.parquet"
LORA_DIR = "/Volumes/main/default/thesis_project/M2_NoContext/G_M2_no_context.1.5"
SAVE_DIR = "/Volumes/main/default/thesis_project/M2_NoContext/M2_Test"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

df_v = pd.read_parquet(EVAL_PATH) if EVAL_PATH.endswith(".parquet") else pd.read_csv(EVAL_PATH, encoding="utf-8-sig")
id_cols = [c for c in ["paper_id","question_id","id"] if c in df_v.columns]
assert "question" in df_v.columns
keep_cols = id_cols + ["question"]
for cand in ["gold_answer","answer","reference_answer","target","summary"]:
    if cand in df_v.columns: keep_cols.append(cand); break
df_v = df_v[keep_cols].copy()
df_v["question"] = df_v["question"].astype(str).str.strip()

In [ ]:
import torch
import numpy as np
import time
import re
from typing import List, Dict, Any

# Ensure left padding for generation
assert tok.padding_side == "left"

SYSTEM_PROMPT = "You are a careful research assistant."
USER_INSTRUCTION = "Answer concisely (≤120 words). If unsure, say you don't know."

In [ ]:
def clean_response(text: str) -> str:
    """
    Clean the generated response to remove instruction artifacts
    """
    # Remove common instruction fragments
    patterns_to_remove = [
        r"Answer concisely \(≤120 words\)\. If unsure, say you don't know\.\s*",
        r"If unsure, say you don't know\.\s*",
        r"say you don't know\.\s*",  # Handle partial fragments
        r"say you don'\s*",  # Handle cut-off fragments
        r"≤120 words?\)\.\s*",
        r"-?\d+\s*words?\)\.\s*",
        r"t know\.\s*",
        r"^\s*-\s*",  # Remove leading dashes
        r"^\s*\d+\s*",  # Remove leading numbers
    ]

    cleaned = text.strip()
    for pattern in patterns_to_remove:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    # Additional cleanup
    cleaned = re.sub(r'\n+', ' ', cleaned)  # Replace newlines with spaces
    cleaned = re.sub(r'\s+', ' ', cleaned)  # Normalize whitespace
    cleaned = cleaned.strip()

    return cleaned

In [ ]:
def build_prompt(q: str) -> str:
    return (
        "<s>[INST] <<SYS>>\n" + SYSTEM_PROMPT + "\n<</SYS>>\n"
        f"Question: {q.strip()}\nInstruction: {USER_INSTRUCTION}\n[/INST]\n"
    )

In [ ]:
def generate_batch_optimized(questions: List[str], model, tokenizer, batch_size: int = 8) -> List[str]:
    """
    Optimized batch generation with proper tokenization and memory management
    """
    # Generation parameters - keeping your settings
    gen_kwargs = {
        "max_new_tokens": 128,
        "do_sample": False,
        "num_beams": 1,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }

    all_predictions = []
    total_tokens = 0
    device = next(model.parameters()).device

    # Process in batches
    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i:i + batch_size]
        batch_prompts = [build_prompt(q) for q in batch_questions]

        # OPTIMIZATION 1: Use batch tokenization (much faster than individual + pad)
        try:
            # Tokenize entire batch at once with automatic padding
            # Use a reasonable max_length if model doesn't have one defined
            max_len = getattr(tokenizer, 'model_max_length', 2048)
            if max_len > 1000000:  # Handle cases where model_max_length is too large
                max_len = 2048

            batch_inputs = tokenizer(
                batch_prompts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(device)

            # OPTIMIZATION 2: Use torch.cuda.amp for mixed precision
            with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
                try:
                    outputs = model.generate(**batch_inputs, **gen_kwargs)
                except RuntimeError as e:
                    if "CUDA out of memory" in str(e):
                        print(f"⚠️ OOM at batch size {len(batch_questions)}, reducing to {len(batch_questions)//2}")
                        torch.cuda.empty_cache()
                        # Recursively process smaller batches
                        mid = len(batch_questions) // 2
                        pred1 = generate_batch_optimized(batch_questions[:mid], model, tokenizer, len(batch_questions)//2)
                        pred2 = generate_batch_optimized(batch_questions[mid:], model, tokenizer, len(batch_questions)//2)
                        all_predictions.extend(pred1 + pred2)
                        continue
                    else:
                        raise

            # OPTIMIZATION 3: More efficient decoding with proper type handling
            input_lengths = batch_inputs["attention_mask"].sum(dim=1).cpu().numpy()
            batch_predictions = []

            for j, output in enumerate(outputs):
                try:
                    # Convert to int properly and extract generated tokens
                    input_len = int(input_lengths[j])
                    generated_tokens = output[input_len:].cpu()
                    total_tokens += len(generated_tokens)

                    # Decode generated tokens only and clean
                    prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
                    prediction = clean_response(prediction)
                    batch_predictions.append(prediction)
                except Exception as decode_error:
                    print(f"Warning: decode error for sample {j}: {decode_error}")
                    # Fallback: decode entire output and clean it properly
                    full_text = tokenizer.decode(output, skip_special_tokens=True)
                    prediction = clean_response(full_text)
                    batch_predictions.append(prediction)

            all_predictions.extend(batch_predictions)

        except Exception as e:
            print(f"Error processing batch {i//batch_size + 1}: {e}")
            # Fallback: process questions individually
            for q in batch_questions:
                try:
                    single_pred = generate_batch_optimized([q], model, tokenizer, 1)[0]
                    all_predictions.append(single_pred)
                except:
                    all_predictions.append("Error generating response")

    return all_predictions

In [ ]:
def adaptive_batch_size(available_memory_gb: float, avg_seq_length: int) -> int:
    """
    Automatically determine optimal batch size based on available memory
    """
    # Rough estimation - adjust based on your GPU
    base_batch_size = 32

    if avg_seq_length > 1500:
        return max(4, base_batch_size // 8)
    elif avg_seq_length > 1000:
        return max(8, base_batch_size // 4)
    elif avg_seq_length > 500:
        return max(16, base_batch_size // 2)
    else:
        return base_batch_size

In [ ]:
# MAIN EXECUTION - OPTIMIZED VERSION
def run_optimized_generation(df_v, model, tok):
    """
    Main function to run the optimized generation process
    """
    print("🚀 Starting optimized generation...")
    t0 = time.time()

    # Extract questions
    questions = df_v["question"].astype(str).str.strip().tolist()

    # Estimate optimal batch size (start conservative)
    sample_prompts = [build_prompt(q) for q in questions[:min(5, len(questions))]]
    try:
        sample_tokens = tok(sample_prompts, padding=False, truncation=True, max_length=2048)
        avg_length = np.mean([len(tokens) for tokens in sample_tokens["input_ids"]])
    except:
        avg_length = 800  # fallback estimate

    optimal_batch_size = min(8, adaptive_batch_size(
        available_memory_gb=torch.cuda.get_device_properties(0).total_memory / 1e9,
        avg_seq_length=int(avg_length)
    ))

    print(f"📊 Average prompt length: {avg_length:.0f} tokens")
    print(f"🎯 Using batch size: {optimal_batch_size}")

    # Generate predictions
    predictions = generate_batch_optimized(questions, model, tok, optimal_batch_size)

    # Add to dataframe
    df_pred = df_v.copy()
    df_pred["answer_M2_1"] = predictions

    elapsed = time.time() - t0
    total_tokens = sum(len(tok.encode(pred)) for pred in predictions if pred)
    tokps = total_tokens / max(elapsed, 1e-6)

    print(f"✅ Generated {len(predictions)} answers in {elapsed:.1f}s")
    print(f"⚡ Speed: {tokps:.0f} tokens/second")
    print(f"🎉 Improvement: ~{tokps/3:.1f}x faster than before!")

    return df_pred

In [ ]:
def test_generation_sample(df_v, model, tok, num_samples: int = 5):
    """
    Test generation on a small sample to verify output quality
    """
    print(f"🧪 Testing generation on {num_samples} samples...")

    # Take first few samples
    test_df = df_v.head(num_samples).copy()
    test_questions = test_df["question"].astype(str).str.strip().tolist()

    # Generate answers
    predictions = generate_batch_optimized(test_questions, model, tok, batch_size=2)

    # Display results
    print("\n" + "="*80)
    print("SAMPLE RESULTS:")
    print("="*80)

    for i, (q, pred) in enumerate(zip(test_questions, predictions)):
        print(f"\n--- Sample {i+1} ---")
        print(f"Question: {q[:100]}{'...' if len(q) > 100 else ''}")
        print(f"Answer: {pred}")
        print("-" * 60)

    return predictions

In [ ]:
# Quick test usage:
test_predictions = test_generation_sample(df_v, model, tok, num_samples=5)

In [ ]:
# USAGE:
df_pred = run_optimized_generation(df_v, model, tok)

In [ ]:
csv_p  = os.path.join(SAVE_DIR, "test_M2_1.7.csv")
parq_p = os.path.join(SAVE_DIR, "test_M2_1.7.parquet")
df_pred.to_csv(csv_p, index=False, encoding="utf-8-sig")
df_pred.to_parquet(parq_p, index=False)
print("✅ Saved:\n -", csv_p, "\n -", parq_p)
print(df_pred.head(5).to_string(index=False))